# IMPORT & LOAD DATA

In [ ]:
import pandas as pd
import numpy as np

In [17]:
orders = pd.read_csv('E:/Tugas Anthar/PROJECT/Sales Performance Dashboard for E-commerce Retail/data/Orders.csv', sep=";",encoding="latin1", decimal=",", engine="python")
returns = pd.read_csv('E:/Tugas Anthar/PROJECT/Sales Performance Dashboard for E-commerce Retail/data/Returns.csv', sep=";",encoding="latin1", decimal=",", engine="python")
person = pd.read_csv('E:/Tugas Anthar/PROJECT/Sales Performance Dashboard for E-commerce Retail/data/Person.csv', sep=";",encoding="latin1", decimal=",", engine="python")

# PRE PROCESSING

In [18]:
df = (orders.merge(returns, on='Order ID', how='left')
        .merge(person, on='Region', how='left'))

In [19]:
person_clean = person[['Region', 'Person']].drop_duplicates(subset='Region')
returns_clean = returns[['Order ID', 'Returned']].drop_duplicates(subset='Order ID')

df = (
    orders
    .merge(person_clean, on='Region', how='left')
    .merge(returns_clean, on='Order ID', how='left')
)

In [20]:
df['Returned'] = df['Returned'].fillna('No')

In [21]:
df = df.rename(columns={
    'Order Date': 'order_date',
    'Ship Date': 'ship_date',
    'Ship Mode': 'ship_mode',
    'Customer ID': 'customer_id',
    'Customer Name': 'customer_name',
    'Country/Region': 'country',
    'Postal Code': 'postal_code',
    'Product ID': 'product_id',
    'Sub-Category': 'sub_category',
    'Product Name': 'product_name',
    'Order ID': 'order_id',
    'Row ID': 'row_id'
})

In [22]:
df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'Segment', 'country', 'City', 'State',
       'postal_code', 'Region', 'product_id', 'Category', 'sub_category',
       'product_name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Person',
       'Returned'],
      dtype='object')

In [23]:
a = sum(df['Sales'])
a

2297200.8603

In [24]:
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True, errors='coerce')
df['ship_date'] = pd.to_datetime(df['ship_date'], dayfirst=True, errors='coerce')


In [25]:
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'postal_code']

for col in numeric_cols:
    if col in df.columns:
        if df[col].dtype == 'object':
            df[col] = (
                df[col]
                .astype(str)
                .str.replace('.', '', regex=False)
                .str.replace(',', '.', regex=False)
            )
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [26]:
df.isnull().sum()

row_id            0
order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_id       0
customer_name     0
Segment           0
country           0
City              0
State             0
postal_code      11
Region            0
product_id        0
Category          0
sub_category      0
product_name      0
Sales             0
Quantity          0
Discount          0
Profit            0
Person            0
Returned          0
dtype: int64

In [27]:
if 'postal_code' in df.columns:
    df['postal_code'] = df['postal_code'].astype('object').astype(str)
df['postal_code'] = df['postal_code'].replace('nan', np.nan).fillna('Unknown')

In [28]:
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_month_name'] = df['order_date'].dt.month_name()
df['order_quarter'] = df['order_date'].dt.quarter
df['year_month'] = df['order_date'].dt.to_period('M').astype(str)
df['ship_days'] = (df['ship_date'] - df['order_date']).dt.days

In [29]:
df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'Segment', 'country', 'City', 'State',
       'postal_code', 'Region', 'product_id', 'Category', 'sub_category',
       'product_name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Person',
       'Returned', 'order_year', 'order_month', 'order_month_name',
       'order_quarter', 'year_month', 'ship_days'],
      dtype='object')

In [33]:
df.to_csv("E:/Tugas Anthar/PROJECT/Sales Performance Dashboard for E-commerce Retail/Data/clean/data_clean.csv")